# SpAM Pilot Analysis

Descriptive analysis of pilot data collected via the SpAM (Spatial Arrangement Method) task.
Data directory: `data/` (local only, gitignored) — flat directory, cohort determined from each
file's own content (`deployment_mode`), not from which subdirectory it lives in.

In [1]:
import sys
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[2]  # analysis/_retired/pilot/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import plotly.io as pio

pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

from analysis.utils.null_distances import simulate, trial_ks_distance
from analysis.utils.parser import parse_pairwise_distances, parse_raw_data
from analysis.utils.stats import mwu_family, print_comparison_line
from analysis._retired.pilot.figures import (
    fig_completion_status,
    fig_demographics,
    fig_trial_duration,
    fig_moves,
    fig_reliability,
    fig_duration_vs_moves,
    fig_within_subject_variability,
    fig_pairwise_distance_distribution,
    fig_reliability_vs_distance,
    fig_move_temporal_profile,
    subject_snr,
)

## 0. Load data

In [2]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    data = parse_raw_data(REPO_ROOT / "data")

df_participants = data["participants"]
df_trials = data["trials"].merge(
    df_participants[["participant_id", "task_version", "sort_area_width", "sort_area_height"]],
    on="participant_id", how="left",
)

for w in caught_warnings:
    print(f"[{w.category.__name__}] {w.message}")

print(f"\nParticipants dataframe: {df_participants.shape[0]} rows x {df_participants.shape[1]} cols")
print(f"Trials dataframe: {df_trials.shape[0]} rows x {df_trials.shape[1]} cols")
print(df_participants["status"].value_counts().to_string())


Participants dataframe: 280 rows x 26 cols
Trials dataframe: 5252 rows x 20 cols
status
full data          203
revoked consent     34
screened out        18
false_positive      15
missing data        10


Versions present in this dataset (exact minor version, not merged): **v1.0**, **v2.0**, **v3.0**,
**v3.06**, **v4.0**, **v4.01**. Figures below merge sub-versions onto their major version (v3.0/v3.06 -> "v3")
for display; the `summary` table right below keeps the full minor-version breakdown. This cell is
manually maintained -- update it if a new minor version is added to the data.

In [3]:
main_trials = df_trials[~df_trials["is_catch"]].assign(
    rt_s=lambda d: d["rt"] / 1000,
    qc_flag_int=lambda d: d["qc_flag"].astype(int),
)

per_version_trials = (
    main_trials
    .groupby("task_version")
    .agg(
        rt_s_mean=("rt_s", "mean"),
        rt_s_median=("rt_s", "median"),
        rt_s_sd=("rt_s", "std"),
        rt_s_min=("rt_s", "min"),
        rt_s_max=("rt_s", "max"),
        moves_mean=("num_moves", "mean"),
        moves_median=("num_moves", "median"),
        moves_sd=("num_moves", "std"),
        qc_flag_rate=("qc_flag_int", "mean"),
    )
)

reliability_by_version = (
    df_trials[df_trials["reliability"].notna()]
    .groupby("task_version")["reliability"]
    .median()
    .rename("reliability")
)

participant_counts = (
    df_participants
    .dropna(subset=["task_version"])
    .groupby("task_version")
    .agg(
        total_N=("participant_id", "count"),
        analytic_N=("status", lambda s: int((s == "full data").sum())),
    )
)

summary = (
    participant_counts
    .join(per_version_trials)
    .join(reliability_by_version)
    .T
    .rename(columns=lambda v: f"v{v:g}")
    .round(2)
)
summary

task_version,v1,v2,v3,v3.06,v4,v4.01
total_N,16.00,10.00,12.00,14.00,41.00,154.00
analytic_N,16.00,10.00,11.00,11.00,34.00,121.00
rt_s_mean,104.33,73.86,78.53,81.83,82.70,87.19
rt_s_median,86.50,65.36,64.86,68.58,70.42,72.77
rt_s_sd,94.61,22.27,40.64,37.22,37.38,57.07
rt_s_min,0.94,60.59,60.04,60.09,60.24,59.57
rt_s_max,926.99,227.23,395.10,407.50,443.92,2292.91
moves_mean,33.85,28.06,28.49,25.37,28.19,26.14
moves_median,24.00,27.00,26.50,23.00,26.00,24.00
moves_sd,24.85,7.96,12.57,10.18,10.76,10.41


`total_N` = all participants with a resolvable session file for that version (participants who
revoked consent or have no file at all can't be attributed to any version, so they're outside this
table entirely -- see `df_participants["status"]` for the full picture). `analytic_N` = the subset
with `status == "full data"` (no screen-out, no missing data). `reliability` = median test-retest
Spearman r across that version's repeat trials (NaN for v1/v2, which predate the trial-repeat
mechanism).

## 1. Completion status

In [4]:
fig_completion_status(df_participants).show()

## 2. Trial duration

In [5]:
fig_trial_duration(df_trials).show()

## 3. Number of moves

In [6]:
fig_moves(df_trials).show()

## 4. Reliability

In [7]:
fig_reliability(df_trials).show()

## 5. Pairwise distance distribution

In [8]:
# Shared null: same K as images_per_trial, enough trials for a stable reference
null_distances = simulate(num_dots=20, num_trials=1000, seed=42)
print(f"Null: {len(null_distances):,} distances  mean={null_distances.mean():.3f}  sd={null_distances.std():.3f}")

fig_pairwise_distance_distribution(df_trials, null_distribution=null_distances).show()

Null: 190,000 distances  mean=0.369  sd=0.175


## 6. Reliability vs. distance

In [9]:
fig_reliability_vs_distance(df_trials).show()

## 7. Trial duration vs. number of moves

In [10]:
fig_duration_vs_moves(df_trials).show()

## 8. Temporal engagement

Do subjects work throughout the trial, or front-load moves and sit idle?

- **Left**: cumulative move fraction vs. time fraction within trial. Diagonal = uniform activity; curve bending top-left = front-loading.
- **Right**: average move rate (moves/s, 5 s bins) over absolute time. Dashed line = 60 s (V2 timer unlock).

In [11]:
fig_move_temporal_profile(df_trials).show()

## 9. Within-subject variability and reliability

In [12]:
fig_within_subject_variability(df_trials).show()

## 10. Participant demographics

In [13]:
fig_demographics(df_participants).show()

## 11. Cohort comparisons

Pairwise Mann-Whitney U tests between task versions, on per-subject aggregates (one value
per subject, avoiding pseudo-replication across trials). With more than two cohorts present,
every pairwise comparison within a metric is run and p-values are corrected across that
family (default Bonferroni; configurable via `mwu_family(..., correction=...)` -- any method
string accepted by `statsmodels.stats.multitest.multipletests` works, e.g. `"holm"`, `"fdr_bh"`).

Six comparisons, ordered to match the figures above:
1. Trial duration (s)
2. Moves per trial
3. KS distance from null -- per-trial D, averaged per subject
4. Idle tail fraction -- (RT - t_last_move) / RT
5. SNR (sigma_d / mean|delta d|) -- **v1 vs v2 only**, see note before that cell. Moved last: with
   the trial-repeat mechanism now covering v3+, this v1/v2-specific measure is the least central
   comparison in this section.
6. Reliability (Spearman r) across v3+ minor versions -- new, see note before that cell.

Effect size is rank-biserial *r* = 1 - 2U / (n_A x n_B); |r| >= 0.5 is conventionally large.

In [14]:
import json

import numpy as np
import pandas as pd


def _mwu_family(df, value_col, group_col="task_version", correction="bonferroni", alpha=0.05):
    """Pairwise Mann-Whitney U for every pair of groups in group_col, with a multiple-
    comparison correction applied across the family -- thin wrapper around
    analysis.utils.stats.mwu_family(), reproducing this notebook's original "v{v:g}"
    comparison labels via label_fn."""
    return mwu_family(
        df, value_col, group_col, correction=correction, alpha=alpha,
        label_fn=lambda v: f"v{v:g}",
    )


def _print_family(result: pd.DataFrame, alpha: float = 0.05) -> None:
    """One line per pairwise comparison, via analysis.utils.stats.print_comparison_line()."""
    for comparison, row in result.iterrows():
        print_comparison_line({**row.to_dict(), "comparison": comparison}, alpha=alpha, corrected=True)

### Test 1 — Trial duration

In [15]:
subj_rt = (
    df_trials.assign(rt_s=df_trials["rt"] / 1000)
    .groupby(["participant_id", "task_version"])["rt_s"]
    .mean()
    .reset_index()
)

result_rt = _mwu_family(subj_rt, "rt_s", correction="bonferroni")
_print_family(result_rt)
result_rt

  v1 vs v2: U=117.0  p_corrected=0.8158 (not significant)  r=-0.463
  v1 vs v3: U=119.0  p_corrected=1.0000 (not significant)  r=-0.352
  v1 vs v3.06: U=104.0  p_corrected=1.0000 (not significant)  r=-0.182
  v1 vs v4: U=383.0  p_corrected=1.0000 (not significant)  r=-0.197
  v1 vs v4.01: U=1340.0  p_corrected=1.0000 (not significant)  r=-0.132
  v2 vs v3: U=47.0  p_corrected=1.0000 (not significant)  r=0.145
  v2 vs v3.06: U=22.0  p_corrected=0.3316 (not significant)  r=0.600
  v2 vs v4: U=96.0  p_corrected=0.1810 (not significant)  r=0.520
  v2 vs v4.01: U=314.0  p_corrected=0.0357 (significant)  r=0.576
  v3 vs v3.06: U=25.0  p_corrected=0.3232 (not significant)  r=0.587
  v3 vs v4: U=110.0  p_corrected=0.1823 (not significant)  r=0.500
  v3 vs v4.01: U=398.0  p_corrected=0.0720 (not significant)  r=0.511
  v3.06 vs v4: U=223.0  p_corrected=1.0000 (not significant)  r=-0.014
  v3.06 vs v4.01: U=770.0  p_corrected=1.0000 (not significant)  r=0.054
  v4 vs v4.01: U=2737.0  p_corrected

,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p,r
comparison,,,,,,,,,,,,
v1 vs v2,16,10,93.9326,61.4024,94.6769,65.6550,59.4445,11.9926,117.0,0.815847,0.054390,-0.4625
v1 vs v3,16,11,93.9326,62.0040,94.6769,69.3758,59.4445,21.3218,119.0,1.000000,0.132307,-0.3523
v1 vs v3.06,16,11,93.9326,73.7994,94.6769,75.6304,59.4445,9.7991,104.0,1.000000,0.444350,-0.1818
v1 vs v4,16,40,93.9326,73.9568,94.6769,76.9891,59.4445,15.3733,383.0,1.000000,0.256981,-0.1969
v1 vs v4.01,16,148,93.9326,74.6214,94.6769,82.4956,59.4445,24.2253,1340.0,1.000000,0.388818,-0.1318
v2 vs v3,10,11,61.4024,62.0040,65.6550,69.3758,11.9926,21.3218,47.0,1.000000,0.597406,0.1455
v2 vs v3.06,10,11,61.4024,73.7994,65.6550,75.6304,11.9926,9.7991,22.0,0.331556,0.022104,0.6000
v2 vs v4,10,40,61.4024,73.9568,65.6550,76.9891,11.9926,15.3733,96.0,0.180972,0.012065,0.5200
v2 vs v4.01,10,148,61.4024,74.6214,65.6550,82.4956,11.9926,24.2253,314.0,0.035663,0.002378,0.5757


### Test 2 — Moves per trial

In [16]:
subj_moves = (
    df_trials
    .groupby(["participant_id", "task_version"])["num_moves"]
    .mean()
    .reset_index()
)

result_moves = _mwu_family(subj_moves, "num_moves", correction="bonferroni")
_print_family(result_moves)
result_moves

  v1 vs v2: U=82.0  p_corrected=1.0000 (not significant)  r=-0.025
  v1 vs v3: U=91.0  p_corrected=1.0000 (not significant)  r=-0.034
  v1 vs v3.06: U=102.0  p_corrected=1.0000 (not significant)  r=-0.159
  v1 vs v4: U=320.5  p_corrected=1.0000 (not significant)  r=-0.002
  v1 vs v4.01: U=1312.0  p_corrected=1.0000 (not significant)  r=-0.108
  v2 vs v3: U=57.0  p_corrected=1.0000 (not significant)  r=-0.036
  v2 vs v3.06: U=66.0  p_corrected=1.0000 (not significant)  r=-0.200
  v2 vs v4: U=182.0  p_corrected=1.0000 (not significant)  r=0.090
  v2 vs v4.01: U=843.0  p_corrected=1.0000 (not significant)  r=-0.139
  v3 vs v3.06: U=70.0  p_corrected=1.0000 (not significant)  r=-0.157
  v3 vs v4: U=201.0  p_corrected=1.0000 (not significant)  r=0.086
  v3 vs v4.01: U=909.0  p_corrected=1.0000 (not significant)  r=-0.117
  v3.06 vs v4: U=166.0  p_corrected=1.0000 (not significant)  r=0.245
  v3.06 vs v4.01: U=758.0  p_corrected=1.0000 (not significant)  r=0.069
  v4 vs v4.01: U=3541.0  p_co

,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p,r
comparison,,,,,,,,,,,,
v1 vs v2,16,10,23.6250,24.8750,30.4948,25.2000,18.6243,5.3862,82.0,1.0000,0.936944,-0.0250
v1 vs v3,16,11,23.6250,25.9167,30.4948,25.9735,18.6243,9.6067,91.0,1.0000,0.901802,-0.0341
v1 vs v3.06,16,11,23.6250,22.1667,30.4948,23.5114,18.6243,7.3679,102.0,1.0000,0.505169,-0.1591
v1 vs v4,16,40,23.6250,25.8600,30.4948,26.6544,18.6243,7.5330,320.5,1.0000,1.000000,-0.0016
v1 vs v4.01,16,148,23.6250,23.0556,30.4948,24.4042,18.6243,7.3065,1312.0,1.0000,0.479813,-0.1081
v2 vs v3,10,11,24.8750,25.9167,25.2000,25.9735,5.3862,9.6067,57.0,1.0000,0.915878,-0.0364
v2 vs v3.06,10,11,24.8750,22.1667,25.2000,23.5114,5.3862,7.3679,66.0,1.0000,0.459526,-0.2000
v2 vs v4,10,40,24.8750,25.8600,25.2000,26.6544,5.3862,7.5330,182.0,1.0000,0.671239,0.0900
v2 vs v4.01,10,148,24.8750,23.0556,25.2000,24.4042,5.3862,7.3065,843.0,1.0000,0.464187,-0.1392


### Test 3 — KS distance from null

D is computed per trial against the shared null (see section 5), then averaged per subject —
the same aggregation as `fig_pairwise_distance_distribution` above.

In [17]:
# null_distances defined in the figures section above (section 5)
df_trials_ks = df_trials.assign(
    ks_D=df_trials["pairwise_distances"].apply(trial_ks_distance, null_distribution=null_distances)
)
subj_ks = (
    df_trials_ks
    .groupby(["participant_id", "task_version"])["ks_D"]
    .mean()
    .reset_index()
)

result_ks = _mwu_family(subj_ks, "ks_D", correction="bonferroni")
_print_family(result_ks)
result_ks

  v1 vs v2: U=38.0  p_corrected=0.4309 (not significant)  r=0.525
  v1 vs v3: U=95.0  p_corrected=1.0000 (not significant)  r=-0.080
  v1 vs v3.06: U=68.0  p_corrected=1.0000 (not significant)  r=0.227
  v1 vs v4: U=513.0  p_corrected=0.0072 (significant)  r=-0.603
  v1 vs v4.01: U=1799.0  p_corrected=0.0099 (significant)  r=-0.519
  v2 vs v3: U=81.0  p_corrected=1.0000 (not significant)  r=-0.473
  v2 vs v3.06: U=66.0  p_corrected=1.0000 (not significant)  r=-0.200
  v2 vs v4: U=355.0  p_corrected=0.0027 (significant)  r=-0.775
  v2 vs v4.01: U=1274.0  p_corrected=0.0021 (significant)  r=-0.722
  v3 vs v3.06: U=43.0  p_corrected=1.0000 (not significant)  r=0.289
  v3 vs v4: U=366.0  p_corrected=0.0129 (significant)  r=-0.664
  v3 vs v4.01: U=1245.0  p_corrected=0.0522 (not significant)  r=-0.529
  v3.06 vs v4: U=380.0  p_corrected=0.0039 (significant)  r=-0.727
  v3.06 vs v4.01: U=1323.0  p_corrected=0.0084 (significant)  r=-0.625
  v4 vs v4.01: U=2866.0  p_corrected=1.0000 (not signi

,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p,r
comparison,,,,,,,,,,,,
v1 vs v2,16,10,0.2679,0.2957,0.2772,0.3037,0.0357,0.0392,38.0,0.430873,0.028725,0.5250
v1 vs v3,16,11,0.2679,0.2623,0.2772,0.2865,0.0357,0.0562,95.0,1.000000,0.748399,-0.0795
v1 vs v3.06,16,11,0.2679,0.2758,0.2772,0.3124,0.0357,0.0788,68.0,1.000000,0.335922,0.2273
v1 vs v4,16,40,0.2679,0.2344,0.2772,0.2396,0.0357,0.0408,513.0,0.007209,0.000481,-0.6031
v1 vs v4.01,16,148,0.2679,0.2317,0.2772,0.2477,0.0357,0.0628,1799.0,0.009907,0.000660,-0.5194
v2 vs v3,10,11,0.2957,0.2623,0.3037,0.2865,0.0392,0.0562,81.0,1.000000,0.072549,-0.4727
v2 vs v3.06,10,11,0.2957,0.2758,0.3037,0.3124,0.0392,0.0788,66.0,1.000000,0.459672,-0.2000
v2 vs v4,10,40,0.2957,0.2344,0.3037,0.2396,0.0392,0.0408,355.0,0.002683,0.000179,-0.7750
v2 vs v4.01,10,148,0.2957,0.2317,0.3037,0.2477,0.0392,0.0628,1274.0,0.002087,0.000139,-0.7216


### Test 4 — Idle tail fraction

In [18]:
def _idle_tail_fraction(row):
    """Fraction of trial time after the last move: (RT - t_last) / RT."""
    try:
        moves = json.loads(row["moves"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    ts = [m["t"] for m in moves if isinstance(m.get("t"), (int, float))]
    if not ts or pd.isna(row["rt"]) or row["rt"] <= 0:
        return np.nan
    return (row["rt"] - max(ts)) / row["rt"]


subj_idle = (
    df_trials
    .assign(idle_tail=df_trials.apply(_idle_tail_fraction, axis=1))
    .groupby(["participant_id", "task_version"])["idle_tail"]
    .mean()
    .reset_index()
)

result_idle = _mwu_family(subj_idle, "idle_tail", correction="bonferroni")
_print_family(result_idle)
result_idle

  v1 vs v2: U=32.0  p_corrected=0.1845 (not significant)  r=0.600
  v1 vs v3: U=32.0  p_corrected=0.0925 (not significant)  r=0.636
  v1 vs v3.06: U=56.0  p_corrected=1.0000 (not significant)  r=0.364
  v1 vs v4: U=104.0  p_corrected=0.0014 (significant)  r=0.675
  v1 vs v4.01: U=409.0  p_corrected=0.0003 (significant)  r=0.655
  v2 vs v3: U=44.0  p_corrected=1.0000 (not significant)  r=0.200
  v2 vs v3.06: U=61.0  p_corrected=1.0000 (not significant)  r=-0.109
  v2 vs v4: U=168.0  p_corrected=1.0000 (not significant)  r=0.160
  v2 vs v4.01: U=680.0  p_corrected=1.0000 (not significant)  r=0.081
  v3 vs v3.06: U=78.0  p_corrected=1.0000 (not significant)  r=-0.289
  v3 vs v4: U=243.0  p_corrected=1.0000 (not significant)  r=-0.104
  v3 vs v4.01: U=931.0  p_corrected=1.0000 (not significant)  r=-0.144
  v3.06 vs v4: U=171.0  p_corrected=1.0000 (not significant)  r=0.223
  v3.06 vs v4.01: U=660.0  p_corrected=1.0000 (not significant)  r=0.189
  v4 vs v4.01: U=3154.0  p_corrected=1.0000 (

,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p,r
comparison,,,,,,,,,,,,
v1 vs v2,16,10,0.0422,0.0680,0.0446,0.1048,0.0176,0.0984,32.0,0.184473,0.012298,0.6000
v1 vs v3,16,11,0.0422,0.1158,0.0446,0.1733,0.0176,0.1637,32.0,0.092517,0.006168,0.6364
v1 vs v3.06,16,11,0.0422,0.0574,0.0446,0.1028,0.0176,0.0991,56.0,1.000000,0.120086,0.3636
v1 vs v4,16,40,0.0422,0.0913,0.0446,0.1225,0.0176,0.0920,104.0,0.001393,0.000093,0.6750
v1 vs v4.01,16,148,0.0422,0.0816,0.0446,0.1162,0.0176,0.0942,409.0,0.000265,0.000018,0.6546
v2 vs v3,10,11,0.0680,0.1158,0.1048,0.1733,0.0984,0.1637,44.0,1.000000,0.459672,0.2000
v2 vs v3.06,10,11,0.0680,0.0574,0.1048,0.1028,0.0984,0.0991,61.0,1.000000,0.698535,-0.1091
v2 vs v4,10,40,0.0680,0.0913,0.1048,0.1225,0.0984,0.0920,168.0,1.000000,0.444875,0.1600
v2 vs v4.01,10,148,0.0680,0.0816,0.1048,0.1162,0.0984,0.0942,680.0,1.000000,0.670916,0.0811


### Test 5 — SNR (v1 vs v2 only)

**Why v3+ is excluded, not generalized:** v1/v2 reliability is measured from individual image
pairs that incidentally recur across distinct trials (a side effect of the old
`unique_images_per_subject` design — different surrounding images each time). v3+ instead
repeats whole trials verbatim (`repeat_of_trial`), holding context constant. These two measures
convolve different sources of variance — v1/v2 includes context effects, v3+ is closer to pure
response noise — so they are not comparable in absolute magnitude (see the subtitle on the
within-subject variability figure above). v3+ is therefore dropped here rather than silently
mixed into the v1-vs-v2 comparison.

**Test 6 below is the "Test 3′" envisioned when this note was first written**: now that v4 also
uses the trial-repeat mechanism, Test 6 compares reliability across v3+ minor versions directly
using the precomputed `reliability` column, without needing this SNR measure at all.

In [19]:
df_trials_v1v2 = df_trials[df_trials["task_version"].isin([1.0, 2.0])]

subj_snr = (
    df_trials_v1v2
    .groupby(["participant_id", "task_version"])
    .apply(subject_snr, include_groups=False)
    .reset_index()
    .rename(columns={0: "snr"})
)

result_snr = _mwu_family(subj_snr, "snr", correction="bonferroni")
_print_family(result_snr)
result_snr

  v1 vs v2: U=95.0  p_corrected=0.4447 (not significant)  r=-0.188


,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p,r
comparison,,,,,,,,,,,,
v1 vs v2,16,10,1.4079,1.2556,1.4134,1.3322,0.3417,0.2152,95.0,0.444738,0.444738,-0.1875


### Test 6 — Reliability across v3+ minor versions

Per-subject median test-retest Spearman r (the `reliability` column, precomputed by
the parser), compared pairwise across v3.0 / v3.06 / v4.0. No formal statistical comparison of
reliability existed before this — only the visual `fig_reliability` above.

In [20]:
subj_reliability = (
    df_trials[df_trials["reliability"].notna()]
    .groupby(["participant_id", "task_version"])["reliability"]
    .median()
    .reset_index()
)
subj_reliability_v3plus = subj_reliability[subj_reliability["task_version"] >= 3.0]

result_reliability = _mwu_family(subj_reliability_v3plus, "reliability", correction="bonferroni")
_print_family(result_reliability)
result_reliability

  v3 vs v3.06: U=36.0  p_corrected=0.6902 (not significant)  r=0.405
  v3 vs v4: U=151.0  p_corrected=0.7002 (not significant)  r=0.314
  v3 vs v4.01: U=601.0  p_corrected=0.8953 (not significant)  r=0.262
  v3.06 vs v4: U=255.0  p_corrected=1.0000 (not significant)  r=-0.159
  v3.06 vs v4.01: U=922.0  p_corrected=1.0000 (not significant)  r=-0.133
  v4 vs v4.01: U=2998.0  p_corrected=1.0000 (not significant)  r=-0.013


,n (A),n (B),median (A),median (B),mean (A),mean (B),std (A),std (B),U,p_corrected,p,r
comparison,,,,,,,,,,,,
v3 vs v3.06,11,11,0.2060,0.2860,0.2077,0.3336,0.1555,0.2191,36.0,0.690210,0.115035,0.4050
v3 vs v4,11,40,0.2060,0.2607,0.2077,0.2694,0.1555,0.1394,151.0,0.700240,0.116707,0.3136
v3 vs v4.01,11,148,0.2060,0.2466,0.2077,0.2771,0.1555,0.1593,601.0,0.895269,0.149211,0.2617
v3.06 vs v4,11,40,0.2860,0.2607,0.3336,0.2694,0.2191,0.1394,255.0,1.000000,0.429470,-0.1591
v3.06 vs v4.01,11,148,0.2860,0.2466,0.3336,0.2771,0.2191,0.1593,922.0,1.000000,0.465607,-0.1327
v4 vs v4.01,40,148,0.2607,0.2466,0.2694,0.2771,0.1394,0.1593,2998.0,1.000000,0.902258,-0.0128


## 12. Quality Control

Validate the catch-trial QC thresholds in `SpAM_Task/task_config.json` against actual collected
data. `computeCatchQcFlag` (`utils.js`) flags a catch trial if **any** of:

- `cluster_mean_distance > catch_trials.cluster_max_mean` (0.15) — images not clustered tightly
- `cluster_sd > catch_trials.cluster_max_sd` (0.10) — cluster too spread (SD of pairwise distances; not a stored field, recomputed below)
- `max_dist_to_target > catch_trials.location_tolerance` (0.20) — worst-case per-image distance to the target corner/centre (mirrors `allImagesNearTarget`; not a stored field, recomputed below)

`cluster_mean_distance` is a stored field; `cluster_sd` and `max_dist_to_target` are recomputed
here from `pairwise_distances` / `final_locations` since the task only saves the boolean QC
outcome, not these intermediate values.

**Note**: the `qc_flag` column reflects whichever thresholds were active in `task_config.json`
*at the time each session ran* — which may differ from the current values above (config has
changed over time). The recomputed metrics below are compared against **today's** thresholds,
which is what answers "are these values still valid going forward."

**Frozen thresholds**: the threshold values hard-coded in the cells below are those that were
in `task_config.json` when this analysis was written. The config has since changed (see this
directory's `KEY_FINDINGS.md`), so re-running this section does not test today's config.

In [21]:
df_catch = df_trials[df_trials["is_catch"]]

# catch_trials QC thresholds as of this analysis (frozen). SpAM_Task/task_config.json has since changed:
# the cluster checks were dropped (catch qc_flag is now always false) and location_tolerance is now 0.15.
_CATCH_THRESH = {
    "cluster_mean_distance": 0.15,   # catch_trials.cluster_max_mean
    "cluster_sd":            0.10,   # catch_trials.cluster_max_sd
    "max_dist_to_target":    0.20,   # catch_trials.location_tolerance
}

_EDGE = 0.15  # fraction from edge for corner targets (utils.js _targetPoint)
_TARGET_FRAC = {
    "center":              (0.50, 0.50),
    "top left corner":     (_EDGE, _EDGE),
    "top right corner":    (1 - _EDGE, _EDGE),
    "bottom left corner":  (_EDGE, 1 - _EDGE),
    "bottom right corner": (1 - _EDGE, 1 - _EDGE),
}


def _cluster_sd(pw_json):
    """Sample SD (ddof=1) of a trial's pairwise distances -- mirrors computeSD() in utils.js.
    Generic across catch and main trials (same pairwise_distances schema)."""
    dists = list(parse_pairwise_distances(pw_json).values())
    return float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0


def _max_dist_to_target(row):
    """Worst-case (max) per-image normalised distance to the catch target -- mirrors allImagesNearTarget() in utils.js."""
    try:
        locs = json.loads(row["final_locations"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    if not locs:
        return np.nan
    fx, fy = _TARGET_FRAC.get(row["catch_trial_target_location"], (0.5, 0.5))
    w, h = row["sort_area_width"], row["sort_area_height"]
    diag = np.sqrt(w**2 + h**2)
    dists = [
        np.sqrt((loc["x"] * w - fx * w) ** 2 + (loc["y"] * h - fy * h) ** 2) / diag
        for loc in locs
    ]
    return max(dists)


df_catch = df_catch.assign(
    cluster_sd=df_catch["pairwise_distances"].apply(_cluster_sd),
    max_dist_to_target=df_catch.apply(_max_dist_to_target, axis=1),
)


def _qc_row(df, thresh_map, direction="max"):
    """
    Per-cohort QC summary row.

    direction="max": metric is capped (flag if value > thresh); shows median/p90/max
                      -- the upper tail is what's relevant for headroom.
    direction="min": metric is floored (flag if value < thresh); shows median/p10/min
                      -- the lower tail is what's relevant for headroom.
    """
    out = {}
    for metric, thresh in thresh_map.items():
        vals = df[metric].dropna()
        if direction == "max":
            out[(metric, "median")]               = round(float(vals.median()), 3)
            out[(metric, "p90")]                  = round(float(vals.quantile(0.9)), 3)
            out[(metric, "max")]                  = round(float(vals.max()), 3)
            out[(metric, f"flagged (>{thresh})")] = round(float((vals > thresh).mean()), 3)
        else:
            out[(metric, "median")]                = round(float(vals.median()), 3)
            out[(metric, "p10")]                   = round(float(vals.quantile(0.1)), 3)
            out[(metric, "min")]                   = round(float(vals.min()), 3)
            out[(metric, f"flagged (<{thresh})")]  = round(float((vals < thresh).mean()), 3)
    out[("recorded", "qc_flag rate")] = round(float(df["qc_flag"].mean()), 3)
    out[("recorded", "n_trials")]     = len(df)
    return pd.Series(out)


rows = {
    f"v{v:g}": _qc_row(df_catch[df_catch["task_version"] == v], _CATCH_THRESH)
    for v in sorted(df_catch["task_version"].unique())
}
rows["all"] = _qc_row(df_catch, _CATCH_THRESH)

qc_table = pd.DataFrame(rows).T
qc_table.columns = pd.MultiIndex.from_tuples(qc_table.columns)
qc_table

cluster_mean_distance                               cluster_sd         \
                     median    p90    max flagged (>0.15)     median    p90   
v1                    0.043  0.112  0.139             0.0      0.022  0.048   
v2                    0.027  0.046  0.073             0.0      0.014  0.025   
v3                    0.045  0.092  0.113             0.0      0.021  0.041   
v3.06                 0.030  0.094  0.142             0.0      0.013  0.040   
v4                    0.018  0.039  0.120             0.0      0.009  0.018   
v4.01                 0.018  0.040  0.119             0.0      0.009  0.020   
all                   0.020  0.055  0.142             0.0      0.010  0.025   

                            max_dist_to_target                               \
         max flagged (>0.1)             median    p90    max flagged (>0.2)   
v1     0.061            0.0              0.115  0.161  0.197            0.0   
v2     0.029            0.0              0.119  0.156  0.183            0.0   
v3     0.049            0.0              0.128  0.162  0.173            0.0   
v3.06  0.079            0.0              0.132  0.173  0.199            0.0   
v4     0.051            0.0              0.098  0.153  0.175            0.0   
v4.01  0.050            0.0              0.098  0.152  0.180            0.0   
all    0.079            0.0              0.107  0.155  0.199            0.0   

          recorded           
      qc_flag rate n_trials  
v1            0.00     32.0  
v2            1.00     20.0  
v3            0.00     44.0  
v3.06         0.00     44.0  
v4            0.00    116.0  
v4.01         0.00    412.0  
all           0.03    668.0

### Experimental (main) trial thresholds

`computeMainQcFlag` (`utils.js`) flags a main trial if **either**:

- `pairwise_sd < quality_control.min_pairwise_distance_sd` (0.04) — images piled in one spot
- `move_ratio < quality_control.min_move_item_ratio` (0.75), where `move_ratio = num_moves / n_items` — too few moves relative to the number of images on screen

`quality_control.min_trial_rt_ms` (60 s) is excluded here since it's UI-enforced (the Done button
is disabled until the floor elapses), not a post-hoc statistical flag.

Both remaining checks are **floors**, the opposite direction from the catch-trial checks above, so
the table below shows the lower tail (median / p10 / min) and flags values *below* threshold.

In [22]:
# main-trial QC thresholds as of this analysis (frozen). SpAM_Task/task_config.json now has
# min_pairwise_distance_sd = 0.1 and min_move_item_ratio = 0.65.
_MAIN_THRESH = {
    "pairwise_sd": 0.04,   # quality_control.min_pairwise_distance_sd
    "move_ratio":  0.75,   # quality_control.min_move_item_ratio
}


def _n_items(final_locations_json):
    try:
        return len(json.loads(final_locations_json))
    except (json.JSONDecodeError, TypeError):
        return np.nan


df_trials_qc = df_trials[~df_trials["is_catch"]].assign(
    pairwise_sd=lambda d: d["pairwise_distances"].apply(_cluster_sd),
    n_items=lambda d: d["final_locations"].apply(_n_items),
)
df_trials_qc["move_ratio"] = df_trials_qc["num_moves"] / df_trials_qc["n_items"]

rows = {
    f"v{v:g}": _qc_row(df_trials_qc[df_trials_qc["task_version"] == v], _MAIN_THRESH, direction="min")
    for v in sorted(df_trials_qc["task_version"].unique())
}
rows["all"] = _qc_row(df_trials_qc, _MAIN_THRESH, direction="min")

qc_table_main = pd.DataFrame(rows).T
qc_table_main.columns = pd.MultiIndex.from_tuples(qc_table_main.columns)
qc_table_main

pairwise_sd                               move_ratio               \
           median    p10    min flagged (<0.04)     median    p10   min   
v1          0.186  0.156  0.098             0.0      1.200  0.700  0.00   
v2          0.173  0.144  0.094             0.0      1.350  0.945  0.70   
v3          0.182  0.138  0.088             0.0      1.325  0.750  0.25   
v3.06       0.181  0.130  0.083             0.0      1.150  0.700  0.25   
v4          0.188  0.158  0.087             0.0      1.300  0.805  0.40   
v4.01       0.192  0.162  0.063             0.0      1.200  0.850  0.00   
all         0.190  0.158  0.063             0.0      1.250  0.850  0.00   

                          recorded           
      flagged (<0.75) qc_flag rate n_trials  
v1              0.119        0.062    160.0  
v2              0.010        0.000    100.0  
v3              0.091        0.091    220.0  
v3.06           0.132        0.132    220.0  
v4              0.049        0.019    852.0  
v4.01           0.038        0.018   3032.0  
all             0.049        0.029   4584.0

### v4.0+ screening pass/fail summary

Quick view of the screening-block outcome, now that the parser preserves the screening_eval
diagnostic fields on `df_participants` (`reasons`, `move_ratio_fail_rate`,
`distance_sd_fail_rate`, `min_reliability`, `median_reliability`).

In [23]:
df_v4 = df_participants[df_participants["task_version"] >= 4.0]

screening_summary = (
    df_v4
    .groupby("task_version")
    .agg(
        n=("participant_id", "count"),
        n_screened_out=("status", lambda s: int((s == "screened out").sum())),
        n_full_data=("status", lambda s: int((s == "full data").sum())),
        screened_out_rate=("status", lambda s: round(float((s == "screened out").mean()), 3)),
    )
)
screening_summary

,n,n_screened_out,n_full_data,screened_out_rate
task_version,,,,
4.00,41,2,34,0.049
4.01,154,16,121,0.104
